# Chapter 1 — Vectors and Linear Combinations

Companion notebook: every figure and timing in the chapter is produced here.

Reproduce with `clae-py -m nbconvert --to notebook --execute --inplace ch01.ipynb`.

## 1.0 — axpy: a pure-Python list comprehension vs NumPy

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [2]:
def by_hand(a, x, y):       # pure Python, a list comprehension over the entries
    return [a * xi + yi for xi, yi in zip(x, y)]


def vectorized(a, x, y):    # NumPy, the whole array at once
    return a * x + y

In [3]:
a = 2.5
rng = np.random.default_rng(0)
x, y = rng.random(10_000_000), rng.random(10_000_000)

t0 = time.perf_counter(); by_hand(a, x, y)
t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); vectorized(a, x, y)
t_vec = time.perf_counter() - t0

print(f'list comprehension: {t_loop:5.2f} s')
print(f'vectorized:         {t_vec * 1e3:5.0f} ms')
print(f'factor:             {t_loop / t_vec:5.0f}x')

list comprehension:  7.48 s
vectorized:            97 ms
factor:                77x


In [4]:
def best(fn, a, x, y, reps):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(a, x, y); ts.append(time.perf_counter() - t0)
    return min(ts)

ns = np.logspace(3, 7, 9).astype(int)
t_loop = [best(by_hand, a, rng.random(n), rng.random(n), 1 if n > 10**6 else 3) for n in ns]
t_vec = [best(vectorized, a, rng.random(n), rng.random(n), 7) for n in ns]

plt.figure(figsize=(7, 5))
plt.semilogx(ns, t_loop, 'o-', label='by_hand (list comprehension)')
plt.semilogx(ns, t_vec, 's-', label='vectorized (NumPy)')
plt.xlabel('n  (length of x and y)'); plt.ylabel('wall-clock time (s)')
plt.title('axpy: scale a vector and add another, loop vs vectorized')
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.savefig('figures/fig_axpy_timing.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.1 — The question, and the unearned answer

Estimation's one sentence: `SalePrice ≈ w1·GrLivArea + w2·OverallQual + …`
— which is axpy. The weights are unknown; the book exists to earn them.
Below, the answer delivered by a function we did not build and do not yet
deserve; by Chapter 11 we will have built it ourselves.

In [5]:
import pandas as pd

zoning  = pd.read_csv('data/zoning.csv')
listing = pd.read_csv('data/listing.csv')
sale    = pd.read_csv('data/sale.csv')          # SalePrice (the target) lives here
housing = pd.merge(zoning, listing, on='Id')
housing = pd.merge(housing, sale, on='Id').set_index('Id')
housing.shape                                    # 1460 homes, 80 features

(1460, 80)

In [6]:
X = housing[['GrLivArea', 'OverallQual']].to_numpy(float)
y = housing['SalePrice'].to_numpy(float)

w, *_ = np.linalg.lstsq(X, y, rcond=None)        # the unearned answer
print('w:', np.round(w, 2))
print(f'house {housing.index[1]}: actual {y[1]:,.0f}   '
      f'predicted {(X @ w)[1]:,.0f}')

w: [   51.87 17604.21]
house 2: actual 181,500   predicted 171,085


## 1.2 — Two operations

In [7]:
def plot_vector(v, color='blue', label=None, **kw):
    plt.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
               color=color, label=label, **kw)

### Scalar multiplication: scaling along the line

In [8]:
v = np.array([2, 1])

plt.figure(figsize=(6, 6))
plt.axhline(0, color='k', lw=0.5); plt.axvline(0, color='k', lw=0.5)
t = np.array([-1.6, 2.3])
plt.plot(t * v[0], t * v[1], '--', color='gray', alpha=0.5)   # the line through the origin
plot_vector(2 * v, 'tab:purple', '2v')
plot_vector(v, 'tab:blue', 'v')
plot_vector(-v, 'tab:red', '-v')
plt.xlim(-4, 5); plt.ylim(-2.5, 3); plt.gca().set_aspect('equal')
plt.grid(True, alpha=0.3); plt.legend(); plt.title('Scalar multiplication')
plt.savefig('figures/fig_scalar_multiplication.png', dpi=150, bbox_inches='tight')
plt.show()

### Vector addition: tip to tail

In [9]:
v1 = np.array([1, 2]); v2 = np.array([3, 1])

plt.figure(figsize=(6, 6))
plt.axhline(0, color='k', lw=0.5); plt.axvline(0, color='k', lw=0.5)
plot_vector(v1, 'tab:blue', 'v1')
plot_vector(v2, 'tab:red', 'v2')
plot_vector(v1 + v2, 'tab:green', 'v1 + v2')
plt.quiver(v1[0], v1[1], v2[0], v2[1], angles='xy', scale_units='xy',
           scale=1, color='tab:red', alpha=0.35)
plt.xlim(-1, 5); plt.ylim(-1, 4); plt.gca().set_aspect('equal')
plt.grid(True, alpha=0.3); plt.legend(); plt.title('Vector addition: tip to tail')
plt.savefig('figures/fig_vector_addition.png', dpi=150, bbox_inches='tight')
plt.show()

### Span: the cloud of all combinations

In [10]:
v = np.array([2, 1]); w = np.array([1, 3])
C, D = np.meshgrid(np.linspace(-2, 2, 25), np.linspace(-2, 2, 25))
span = C.ravel()[:, None] * v + D.ravel()[:, None] * w   # every c*v + d*w

plt.figure(figsize=(6, 6))
plt.scatter(span[:, 0], span[:, 1], s=4, alpha=0.4)
plot_vector(v, 'tab:blue', 'v'); plot_vector(w, 'tab:red', 'w')
plt.gca().set_aspect('equal'); plt.grid(True, alpha=0.3); plt.legend()
plt.title('The span of v and w: every c*v + d*w')
plt.savefig('figures/fig_span_cloud.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.4 — Cauchy-Schwarz, verified at Ames scale

In [11]:
import pandas as pd

zoning = pd.read_csv('data/zoning.csv')
listing = pd.read_csv('data/listing.csv')
sale = pd.read_csv('data/sale.csv')
housing = pd.merge(zoning, listing, on='Id')
housing = pd.merge(housing, sale, on='Id').set_index('Id')

a = housing['GrLivArea'].to_numpy(float)
b = housing['OverallQual'].to_numpy(float)
lhs = abs(a @ b)
rhs = np.linalg.norm(a) * np.linalg.norm(b)
print(f'|a.b|     = {lhs:,.0f}')
print(f'|a||b|    = {rhs:,.0f}')
print(f'lhs/rhs   = {lhs/rhs:.4f}   (must be <= 1)')

|a.b|     = 14,123,976
|a||b|    = 14,645,262
lhs/rhs   = 0.9644   (must be <= 1)
